In [1]:
import sys
sys.path = [p for p in sys.path if 'AppData' not in p]

import sys
print("Python executable:", sys.executable)

import pandas as pd
print("Pandas version:", pd.__version__)
print("Pandas path:", pd.__file__)

Python executable: d:\DuanchungC\.venv\Scripts\python.exe
Pandas version: 2.3.3
Pandas path: d:\DuanchungC\.venv\Lib\site-packages\pandas\__init__.py


In [2]:
# CONFIG: fill in once for your group, both notebooks read the same block
TOPIC = "Dự báo ngày nắng nóng cực đoan và cảnh báo sóng nhiệt nhiều trạm từ NOAA GHCN-Daily"
GROUP = "Nhom 1"                     # Nhóm 9 (theo trang đầu của đề cương)
PRIMARY_SOURCE = {
    "name": "NOAA GHCN-Daily", 
    "url": "https://www.ncei.noaa.gov/products/land-based-station/global-historical-climatology-network-daily", 
    "license": "public domain", 
    "path": "data/raw/ghcnd_tx.parquet"
}
SECOND_SOURCE  = {
    "name": "NWS HeatRisk & Weather Indicators", 
    "url": "https://www.weather.gov/heatrisk/", 
    "license": "public domain", 
    "path": "data/raw/secondary.csv"
}
UNIT_COL   = "STATION"               # Cột định danh trạm khí tượng
TIME_COL   = "DATE"                  # Cột thời gian (ngày)
TARGET_COL = "TMAX"                  # Biến mục tiêu cần dự báo (nhiệt độ tối đa)
EXOG_COLS  = ["TMIN", "PRCP"]        # Cột biến phụ/ngoại sinh: TMIN (nhiệt độ tối thiểu) và PRCP (lượng mưa)
FREQ       = "D"                     # Tần suất dữ liệu dạng ngày ('D')
HORIZONS   = [1, 2, 3]               # Tầm dự báo: 1, 2 và 3 ngày tới
SEASON     = 365                     # Chu kỳ mùa tính bằng số bước (365 ngày cho dữ liệu ngày)
TEST_START = "2024-01-01"            # Mốc thời gian bắt đầu tập kiểm thử (time-based split)
EVENT_QUANTILE = 0.95                # Ngưỡng xác định ngày cực đoan (phân vị 95% TMAX mùa hè mỗi trạm)
print("Topic:", TOPIC); print("Group:", GROUP)

Topic: Dự báo ngày nắng nóng cực đoan và cảnh báo sóng nhiệt nhiều trạm từ NOAA GHCN-Daily
Group: Nhom 1


In [3]:
# AI Audit Log helper: every prompt that changed your work is one row (2 minutes per entry, 3-5 per week)
import pandas as pd, os, datetime as dt
os.makedirs("report", exist_ok=True)
AUDIT_PATH = "report/ai_audit_log.csv"
def audit(step, prompt, tool, ai_output_summary, verified_how, decision, hallucination=False):
    """Append one entry. decision: what you kept / changed / rejected. hallucination=True if the AI answer was wrong and you caught it."""
    row = {"date": dt.date.today().isoformat(), "group": GROUP, "step": step, "prompt": prompt[:500], "tool": tool,
           "ai_output": ai_output_summary[:500], "verified_how": verified_how[:300], "decision": decision[:300], "hallucination": int(hallucination)}
    df = pd.DataFrame([row])
    df.to_csv(AUDIT_PATH, mode="a", header=not os.path.exists(AUDIT_PATH), index=False)
    print("audit entry saved:", step)
# example (delete after reading): audit("Step 2", "Given columns ... how to treat gaps longer than 3 steps?", "Claude", "suggested interpolate(limit=3) then drop", "checked share of gaps > 3 in Q4 below", "kept limit=3, dropped 1.2% rows")

In [4]:
# Step 0a: run once in a terminal (not in the notebook)
# python -m venv .venv && source .venv/bin/activate      (Windows: .venv\Scripts\activate)
# pip install pandas numpy duckdb pyarrow scikit-learn lightgbm statsmodels matplotlib seaborn ydata-profiling mapie shap requests
# pip freeze > requirements.txt
import os
for d in ["data/raw", "data/processed", "sql", "report", "notebooks"]: os.makedirs(d, exist_ok=True)
print("folders ready")

folders ready


In [5]:
# Step 0b: imports and plotting style (English labels in figures, no top/right spines, no in-figure titles)
import pandas as pd, numpy as np, duckdb, json, hashlib, warnings, re
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore"); np.random.seed(42)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
TEAL, ACC, GREY = "#1B6B6D", "#F4A261", "#9FBFBF"
def style(ax): ax.spines[["top", "right"]].set_visible(False)
def savefig(fig, name): fig.tight_layout(); fig.savefig(f"report/{name}.png", dpi=300); plt.close(fig); print("saved report/" + name + ".png")
con = duckdb.connect()
print("pandas", pd.__version__, "duckdb", duckdb.__version__)

pandas 2.3.3 duckdb 1.5.5


In [6]:
# Step 1a: source registry (append every file you download; this becomes Table "Data sources" in the paper)
SOURCES_PATH = "report/data_sources.csv"
def register_source(src, rows=None, cols=None, start=None, end=None, note=""):
    """src: dict with name, url, license, path. rows/cols/start/end are filled after loading."""
    row = {**src, "downloaded": pd.Timestamp.today().date().isoformat(), "rows": rows, "cols": cols, "start": start, "end": end, "note": note}
    df = pd.DataFrame([row]); df.to_csv(SOURCES_PATH, mode="a", header=not os.path.exists(SOURCES_PATH), index=False)
    return row
print("registry at", SOURCES_PATH)

registry at report/data_sources.csv


In [7]:
import pandas as pd

def load_table(path):
    if path.endswith('.parquet'):
        return pd.read_parquet(path)
    else:
        return pd.read_csv(path)

def register_source(source_dict, rows, cols, start, end):
    print(f"Đã đăng ký nguồn: {source_dict['name']} ({rows} dòng, {cols} cột từ {start} đến {end})")

In [8]:
# Tạo dữ liệu mẫu (dummy data) cho NOAA GHCN-Daily để test pipeline
import pandas as pd
import numpy as np
import os

os.makedirs("data/raw", exist_ok=True)

# Tạo danh sách các trạm mẫu và chuỗi thời gian
np.random.seed(42)
stations = [f"USC48{i:04d}" for i in range(1, 11)]
dates = pd.date_range(start="2020-01-01", end="2024-12-31", freq="D")

data = []
for station in stations:
    n_days = len(dates)
    tmax = np.random.normal(loc=30, scale=5, size=n_days) + np.sin(np.linspace(0, 2*np.pi, n_days)) * 8
    tmin = tmax - np.random.uniform(5, 12, size=n_days)
    prcp = np.random.exponential(scale=2, size=n_days) * (np.random.rand(n_days) > 0.7)
    
    df_station = pd.DataFrame({
        "STATION": station,
        "DATE": dates,
        "TMAX": tmax,
        "TMIN": tmin,
        "PRCP": prcp
    })
    data.append(df_station)

df_dummy = pd.concat(data, ignore_index=True)

# Lưu thành file parquet theo đúng đường dẫn CONFIG
dummy_path = "data/raw/ghcnd_tx.parquet"
df_dummy.to_parquet(dummy_path, index=False)
print("Đã tạo xong file dữ liệu mẫu tại:", dummy_path)

# Tiến hành chạy load bảng
raw = load_table(PRIMARY_SOURCE["path"])
raw.columns = [c.strip() for c in raw.columns]
raw[TIME_COL] = pd.to_datetime(raw[TIME_COL], errors="coerce", utc=False)
print("Shape:", raw.shape)
display(raw.head(3))
display(raw.dtypes.value_counts())
register_source(PRIMARY_SOURCE, rows=len(raw), cols=raw.shape[1], start=str(raw[TIME_COL].min()), end=str(raw[TIME_COL].max()))

Đã tạo xong file dữ liệu mẫu tại: data/raw/ghcnd_tx.parquet
Shape: (18270, 5)


,STATION,DATE,TMAX,TMIN,PRCP
0,USC480001,2020-01-01,32.483571,23.327945,10.838215
1,USC480001,2020-01-02,29.336206,24.265288,0.000000
2,USC480001,2020-01-03,33.293498,23.840825,0.000000


float64           3
object            1
datetime64[ns]    1
Name: count, dtype: int64

Đã đăng ký nguồn: NOAA GHCN-Daily (18270 dòng, 5 cột từ 2020-01-01 00:00:00 đến 2024-12-31 00:00:00)


In [9]:
# Step 1b: load the primary source (edit the reader to match your file: csv, parquet, or an API export saved to data/raw)
def load_table(path):
    """Read csv/parquet with DuckDB so large files never need to fit in memory; returns a pandas DataFrame."""
    if path.endswith(".parquet"): return con.execute(f"SELECT * FROM read_parquet('{path}')").df()
    return con.execute(f"SELECT * FROM read_csv_auto('{path}', union_by_name=true, sample_size=-1)").df()

raw = load_table(PRIMARY_SOURCE["path"])
raw.columns = [c.strip() for c in raw.columns]
raw[TIME_COL] = pd.to_datetime(raw[TIME_COL], errors="coerce", utc=False)
print(raw.shape); display(raw.head(3)); display(raw.dtypes.value_counts())
register_source(PRIMARY_SOURCE, rows=len(raw), cols=raw.shape[1], start=str(raw[TIME_COL].min()), end=str(raw[TIME_COL].max()))

(18270, 5)


,STATION,DATE,TMAX,TMIN,PRCP
0,USC480001,2020-01-01,32.483571,23.327945,10.838215
1,USC480001,2020-01-02,29.336206,24.265288,0.000000
2,USC480001,2020-01-03,33.293498,23.840825,0.000000


float64           3
object            1
datetime64[ns]    1
Name: count, dtype: int64

Đã đăng ký nguồn: NOAA GHCN-Daily (18270 dòng, 5 cột từ 2020-01-01 00:00:00 đến 2024-12-31 00:00:00)


In [10]:
# Tạo file dữ liệu thứ hai mẫu (secondary source) để test pipeline
import pandas as pd
import os

os.makedirs("data/raw", exist_ok=True)

# Tạo dữ liệu phụ trợ theo chuỗi thời gian (ví dụ: chỉ số nhiệt độ/mưa hoặc dữ liệu khu vực)
dates = pd.date_range(start="2020-01-01", end="2024-12-31", freq="D")
sec_dummy = pd.DataFrame({
    "DATE": dates,
    "SECONDARY_VAL": np.random.uniform(10, 50, size=len(dates))
})
sec_dummy.to_csv("data/raw/secondary.csv", index=False)
print("Đã tạo xong file secondary.csv mẫu!")

Đã tạo xong file secondary.csv mẫu!


In [11]:
# Step 1c: load the secondary source (covariates); it must share TIME_COL (and UNIT_COL if it is unit-specific)
sec = load_table(SECOND_SOURCE["path"]); sec.columns = [c.strip() for c in sec.columns]
sec[TIME_COL] = pd.to_datetime(sec[TIME_COL], errors="coerce")
SEC_HAS_UNIT = UNIT_COL in sec.columns
print(sec.shape, "unit-specific:", SEC_HAS_UNIT); display(sec.head(3))
register_source(SECOND_SOURCE, rows=len(sec), cols=sec.shape[1], start=str(sec[TIME_COL].min()), end=str(sec[TIME_COL].max()))


(1827, 2) unit-specific: False


,DATE,SECONDARY_VAL
0,2020-01-01,23.022313
1,2020-01-02,20.334230
2,2020-01-03,44.971820


Đã đăng ký nguồn: NWS HeatRisk & Weather Indicators (1827 dòng, 2 cột từ 2020-01-01 00:00:00 đến 2024-12-31 00:00:00)


In [12]:
# Export the standardised raw snapshot to CSV and read it back to confirm the round trip (dtypes and row count)
RAW_CSV = "data/raw/primary_snapshot.csv"
raw.to_csv(RAW_CSV, index=False)
check = pd.read_csv(RAW_CSV, parse_dates=[TIME_COL])
assert len(check) == len(raw), "row count changed in the CSV round trip"
print("wrote", RAW_CSV, "| rows", len(check), "| columns", list(check.columns)[:8])

wrote data/raw/primary_snapshot.csv | rows 18270 | columns ['STATION', 'DATE', 'TMAX', 'TMIN', 'PRCP']


In [13]:
# Step 1d: combine several downloaded chunks (data/raw/primary_*.csv) into one table, deduplicated on unit and time
import glob
chunks = sorted(glob.glob(PRIMARY_SOURCE["path"].replace(".csv", "_*.csv")))
if chunks:
    extra = pd.concat([load_table(f) for f in chunks]); extra[TIME_COL] = pd.to_datetime(extra[TIME_COL], errors="coerce")
    before = len(raw); raw = pd.concat([raw, extra]).drop_duplicates([UNIT_COL, TIME_COL] if UNIT_COL in raw.columns else [TIME_COL])
    print(f"added {len(raw) - before} rows from {len(chunks)} chunks")
else:
    print("no extra chunks found (pattern primary_*.csv); skip if your source is a single file")
if UNIT_COL not in raw.columns: raw[UNIT_COL] = "all"; print("single-series data: UNIT_COL set to 'all'")

added 0 rows from 1 chunks


In [14]:
# Step 2a: source comparison table (đã được tinh chỉnh để chạy an toàn)
import os
os.makedirs("report", exist_ok=True)

def compare_sources(a, b, time_col, common_var=None):
    ta, tb = a[time_col].dropna(), b[time_col].dropna()
    ov_start, ov_end = max(ta.min(), tb.min()), min(ta.max(), tb.max())
    
    # Sử dụng giá trị mặc định 'D' (Daily) nếu biến FREQ chưa được định nghĩa trong CONFIG
    freq_val = globals().get('FREQ', 'D')
    ga = set(ta.dt.floor(freq_val).unique())
    gb = set(tb.dt.floor(freq_val).unique())
    
    rows = [
        ["overlap start", ov_start], 
        ["overlap end", ov_end], 
        ["steps only in primary", len(ga - gb)], 
        ["steps only in secondary", len(gb - ga)],
        ["latest timestamp primary", ta.max()], 
        ["latest timestamp secondary", tb.max()]
    ]
    if common_var and common_var in a.columns and common_var in b.columns:
        m = a.groupby(time_col)[common_var].mean().to_frame("a").join(b.groupby(time_col)[common_var].mean().to_frame("b"), how="inner")
        rows += [["correlation of common variable", round(m.a.corr(m.b), 3)], ["mean difference (primary minus secondary)", round((m.a - m.b).mean(), 3)]]
    return pd.DataFrame(rows, columns=["metric", "value"])

cmp_tbl = compare_sources(raw, sec, TIME_COL, common_var=None)    # set common_var="temp" if both sources carry the same variable
cmp_tbl.to_csv("report/table_source_comparison.csv", index=False)
display(cmp_tbl)

,metric,value
0,overlap start,2020-01-01 00:00:00
1,overlap end,2024-12-31 00:00:00
2,steps only in primary,0
3,steps only in secondary,0
4,latest timestamp primary,2024-12-31 00:00:00
5,latest timestamp secondary,2024-12-31 00:00:00


In [15]:
# Step 2b: cleaning with a log; each step records rows before and after and the reason (đã tối ưu an toàn)
import pandas as pd
import os

os.makedirs("report", exist_ok=True)

# Lấy các biến cấu hình an toàn từ môi trường nếu chưa có
time_col = globals().get('TIME_COL', 'DATE')
unit_col = globals().get('UNIT_COL', 'STATION')
target_col = globals().get('TARGET_COL', 'TMAX')

clean_log = []
def step(df, name, fn, reason):
    n0 = len(df)
    out = fn(df)
    clean_log.append([name, n0, len(out), n0 - len(out), reason])
    return out

df = raw.copy()
df = step(df, "parse time", lambda x: x.dropna(subset=[time_col]), "rows with unparseable timestamps")
df = step(df, "drop duplicates", lambda x: x.drop_duplicates([unit_col, time_col]), "same unit and timestamp")

df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
df = step(df, "drop missing target", lambda x: x.dropna(subset=[target_col]), "target missing")

lo, hi = df[target_col].quantile([0.001, 0.999])
df = step(df, "physical range", lambda x: x[(x[target_col] >= lo) & (x[target_col] <= hi)], f"outside [{lo:.3g}, {hi:.3g}] (0.1% tails)")

log_df = pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"])
log_df.to_csv("report/table_cleaning_log.csv", index=False)
display(log_df)

,step,rows_before,rows_after,rows_removed,reason
0,parse time,18270,18270,0,rows with unparseable timestamps
1,drop duplicates,18270,18270,0,same unit and timestamp
2,drop missing target,18270,18270,0,target missing
3,physical range,18270,18232,38,"outside [8.78, 50.5] (0.1% tails)"


In [16]:
# Step 2c: regular time grid per unit, then short-gap interpolation (đã tối ưu an toàn)
import pandas as pd
import numpy as np

time_col = globals().get('TIME_COL', 'DATE')
unit_col = globals().get('UNIT_COL', 'STATION')
target_col = globals().get('TARGET_COL', 'TMAX')
freq_val = globals().get('FREQ', 'D')

# Lọc ra các cột số khác để resample mean
numeric_cols = [c for c in df.columns if c not in (unit_col, time_col, target_col) and pd.api.types.is_numeric_dtype(df[c])]

df = df.set_index(time_col).groupby(unit_col)[[target_col] + numeric_cols].resample(freq_val).mean().reset_index()

gap = df.groupby(unit_col)[target_col].apply(lambda s: s.isna().mean()).rename("missing_share")

df[target_col] = df.groupby(unit_col)[target_col].transform(lambda s: s.interpolate(limit=3))

n_before = len(df)
df = df.dropna(subset=[target_col])
clean_log.append(["regular grid + interpolate(limit=3)", n_before, len(df), n_before - len(df), "gaps longer than 3 steps dropped"])

print("missing share per unit before interpolation:")
display(gap.describe().round(3))

missing share per unit before interpolation:


count    10.000
mean      0.002
std       0.001
min       0.001
25%       0.001
50%       0.002
75%       0.003
max       0.004
Name: missing_share, dtype: float64

In [17]:
# Step 2d: keep units with enough history (đã tối ưu an toàn)
import pandas as pd
unit_col = globals().get('UNIT_COL', 'STATION')
cnt = df.groupby(unit_col).size()
keep = cnt[cnt >= 0.9 * cnt.max()].index
n_before = len(df)
df = df[df[unit_col].isin(keep)]
clean_log.append(["drop short units", n_before, len(df), n_before - len(df), f"{len(cnt) - len(keep)} units with < 90% coverage"])
print("units kept:", len(keep), "of", len(cnt))

units kept: 10 of 10


In [18]:
# Step 2e: merge the secondary source (đã fix đồng bộ kiểu dữ liệu thời gian)
sec_num = sec[[TIME_COL] + ([UNIT_COL] if SEC_HAS_UNIT else []) + [c for c in EXOG_COLS if c in sec.columns]].copy()
for c in EXOG_COLS:
    if c in sec_num.columns: sec_num[c] = pd.to_numeric(sec_num[c], errors="coerce")

# Đồng bộ kiểu dữ liệu thời gian (datetime64[ns]) cho cả hai bảng để tránh lỗi MergeError
sec_num[TIME_COL] = pd.to_datetime(sec_num[TIME_COL]).astype("datetime64[ns]")
df[TIME_COL] = pd.to_datetime(df[TIME_COL]).astype("datetime64[ns]")

sec_num = sec_num.sort_values(TIME_COL)
df = df.sort_values(TIME_COL)

tol = pd.Timedelta(f"1{FREQ}")
m = pd.merge_asof(df, sec_num, on=TIME_COL, by=UNIT_COL if SEC_HAS_UNIT else None, direction="nearest", tolerance=tol)
share = {c: round(m[c].notna().mean(), 3) for c in EXOG_COLS if c in m.columns}
print("share of rows with each covariate:", share)
df = m.sort_values([UNIT_COL, TIME_COL]).reset_index(drop=True)

share of rows with each covariate: {'TMIN': np.float64(0.998), 'PRCP': np.float64(0.998)}


In [19]:
# Step 2f: save the clean table and the cleaning log; print a one-paragraph data statement for the paper
df.to_parquet("data/processed/clean.parquet", index=False)
df.to_csv("data/processed/clean.csv", index=False)                     # separate clean CSV for submission and for opening in Excel
chk = pd.read_csv("data/processed/clean.csv", parse_dates=[TIME_COL]); assert len(chk) == len(df) and chk[TARGET_COL].notna().all(), "clean.csv round trip failed"
print("wrote data/processed/clean.csv with", len(chk), "rows")
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv("report/table_cleaning_log.csv", index=False)
print(f"Data statement: {PRIMARY_SOURCE['name']} ({PRIMARY_SOURCE['license']}) merged with {SECOND_SOURCE['name']}; {len(df):,} rows, {df[UNIT_COL].nunique()} units, "
      f"{df[TIME_COL].min().date()} to {df[TIME_COL].max().date()} at frequency {FREQ}; {sum(r[3] for r in clean_log):,} rows removed by cleaning.")

wrote data/processed/clean.csv with 18270 rows
Data statement: NOAA GHCN-Daily (public domain) merged with NWS HeatRisk & Weather Indicators; 18,270 rows, 10 units, 2020-01-01 to 2024-12-31 at frequency D; 38 rows removed by cleaning.


In [20]:
# EDA 1: automatic profile (open report/profile.html in a browser) and summary statistics
from ydata_profiling import ProfileReport
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, title=f"Profile {GROUP}", minimal=True).to_file("report/profile.html")
desc = df[[TARGET_COL] + [c for c in EXOG_COLS if c in df.columns]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.round(3)
desc.to_csv("report/table_describe.csv"); display(desc)

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 304.58it/s]


,count,mean,std,min,1%,5%,50%,95%,99%,max
TMAX,18270.0,30.003,7.466,8.825,13.777,17.794,30.076,42.057,46.007,50.498
TMIN,18232.0,21.495,7.735,-2.564,4.635,8.709,21.561,34.029,38.224,44.509
PRCP,18232.0,0.599,1.440,0.000,0.000,0.000,0.000,3.552,6.842,20.977


In [21]:
# EDA 2: distribution of the target (histogram with many bins) and log-scale check for skewed data
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].hist(df[TARGET_COL], bins=60, color=TEAL); axes[0].set_xlabel(TARGET_COL); axes[0].set_ylabel("Count"); style(axes[0])
pos = df[TARGET_COL][df[TARGET_COL] > 0]
axes[1].hist(np.log1p(pos), bins=60, color=ACC); axes[1].set_xlabel("log1p(" + TARGET_COL + ")"); style(axes[1])
savefig(fig, "fig_eda_distribution"); print("skewness:", round(df[TARGET_COL].skew(), 2), "-> consider modelling log1p if skewness > 2")

saved report/fig_eda_distribution.png
skewness: -0.03 -> consider modelling log1p if skewness > 2


In [22]:
# EDA 3: time series of the first four units over the whole period (look for trends, regime changes, gaps)
units = df[UNIT_COL].unique()[:4]
fig, ax = plt.subplots(figsize=(11, 3.4))
for u, col in zip(units, [TEAL, ACC, GREY, "#264653"]):
    g = df[df[UNIT_COL] == u]; ax.plot(g[TIME_COL], g[TARGET_COL], lw=0.7, color=col, label=str(u))
ax.axvline(pd.Timestamp(TEST_START), color="k", ls="--", lw=1); ax.set_ylabel(TARGET_COL); ax.legend(frameon=False, ncol=4); style(ax)
savefig(fig, "fig_eda_timeseries")

saved report/fig_eda_timeseries.png


In [23]:
# EDA 4: calendar seasonality: mean target by hour of day, day of week and month (only the ones that make sense at your FREQ)
t = df[TIME_COL]; cal = pd.DataFrame({"hour": t.dt.hour, "dow": t.dt.dayofweek, "month": t.dt.month, TARGET_COL: df[TARGET_COL]})
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, k in zip(axes, ["hour", "dow", "month"]):
    s = cal.groupby(k)[TARGET_COL].mean(); ax.plot(s.index, s.values, marker="o", color=TEAL); ax.set_xlabel(k); style(ax)
axes[0].set_ylabel("mean " + TARGET_COL); savefig(fig, "fig_eda_seasonality")
season_strength = {k: round(cal.groupby(k)[TARGET_COL].mean().std() / cal[TARGET_COL].std(), 3) for k in ["hour", "dow", "month"]}
print("seasonal strength (std of group means / total std):", season_strength)

saved report/fig_eda_seasonality.png
seasonal strength (std of group means / total std): {'hour': np.float64(nan), 'dow': np.float64(0.008), 'month': np.float64(0.015)}


In [24]:
# EDA 5: heterogeneity across units: mean and variability per unit, sorted (global models must handle this spread)
per_unit = df.groupby(UNIT_COL)[TARGET_COL].agg(["mean", "std", "min", "max", "count"]).sort_values("mean")
per_unit.to_csv("report/table_per_unit.csv")
fig, ax = plt.subplots(figsize=(10, 3.2)); ax.bar(range(len(per_unit)), per_unit["mean"], color=TEAL, yerr=None)
ax.set_xlabel("units sorted by mean"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_units")
print("ratio of largest to smallest unit mean:", round(per_unit["mean"].max() / max(per_unit["mean"].min(), 1e-9), 2))

saved report/fig_eda_units.png
ratio of largest to smallest unit mean: 1.01


In [25]:
# EDA 6: relation with covariates: binned means and Spearman correlation (robust to outliers)
from scipy.stats import spearmanr
rows = []
for c in [c for c in EXOG_COLS if c in df.columns]:
    ok = df[[c, TARGET_COL]].dropna(); rho, p = spearmanr(ok[c], ok[TARGET_COL]); rows.append([c, round(rho, 3), p, len(ok)])
    fig, ax = plt.subplots(figsize=(5.5, 3.2)); b = ok.groupby(pd.qcut(ok[c], 10, duplicates="drop"))[TARGET_COL].mean()
    ax.plot(range(len(b)), b.values, marker="o", color=TEAL); ax.set_xlabel(c + " (deciles)"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_covariate_" + c)
corr_tbl = pd.DataFrame(rows, columns=["covariate", "spearman_rho", "p_value", "n"]); corr_tbl.to_csv("report/table_covariate_corr.csv", index=False); display(corr_tbl)

saved report/fig_eda_covariate_TMIN.png
saved report/fig_eda_covariate_PRCP.png


,covariate,spearman_rho,p_value,n
0,TMIN,0.966,0.000000,18232
1,PRCP,-0.004,0.570274,18232


In [26]:
# EDA 7: cross-correlation with lags: does the covariate lead the target? (positive lag = covariate earlier)
def cross_corr(g, x, y, lags):
    return [g[y].corr(g[x].shift(k)) for k in lags]
lags = list(range(0, 4 * SEASON + 1, max(1, SEASON // 4)))
fig, ax = plt.subplots(figsize=(7, 3.2))
for c in [c for c in EXOG_COLS if c in df.columns][:3]:
    cc = np.nanmean([cross_corr(g, c, TARGET_COL, lags) for _, g in df.groupby(UNIT_COL) if len(g) > 5 * SEASON], axis=0)
    ax.plot(lags, cc, marker="o", label=c); print(c, "best lag:", lags[int(np.nanargmax(np.abs(cc)))], "corr:", round(float(np.nanmax(np.abs(cc))), 3))
ax.axhline(0, color=GREY); ax.set_xlabel("lag (steps, covariate leads target)"); ax.set_ylabel("correlation"); ax.legend(frameon=False); style(ax); savefig(fig, "fig_eda_crosscorr")

TMIN best lag: 0 corr: 0.966
PRCP best lag: 1274 corr: 0.032
saved report/fig_eda_crosscorr.png


In [27]:
# EDA 8: autocorrelation of the target (ACF) averaged over units: which lags to use as features and whether SEASON is right
from statsmodels.tsa.stattools import acf
nl = 2 * SEASON + 1
acfs = [acf(g[TARGET_COL].values, nlags=nl, fft=True) for _, g in df.groupby(UNIT_COL) if len(g) > 4 * SEASON]
acf_mean = np.mean(acfs, axis=0)
fig, ax = plt.subplots(figsize=(8, 3.2)); ax.stem(range(nl + 1), acf_mean, basefmt=" "); ax.set_xlabel("lag (steps)"); ax.set_ylabel("ACF"); style(ax); savefig(fig, "fig_eda_acf")
print("ACF at lag 1:", round(acf_mean[1], 3), "| at SEASON:", round(acf_mean[SEASON], 3), "-> a seasonal naive baseline is strong if ACF at SEASON > 0.7")

saved report/fig_eda_acf.png
ACF at lag 1: 0.56 | at SEASON: 0.219 -> a seasonal naive baseline is strong if ACF at SEASON > 0.7


In [28]:
# EDA 9: missingness pattern over time per unit (heatmap of monthly missing share) and regime check (rolling mean over time)
miss = df.set_index(TIME_COL).groupby(UNIT_COL)[TARGET_COL].resample("MS").apply(lambda s: s.isna().mean()).unstack(0)
fig, ax = plt.subplots(figsize=(10, 3.4)); sns.heatmap(miss.T, cmap="Blues", cbar_kws={"label": "missing share"}, ax=ax); ax.set_xlabel("month"); ax.set_ylabel(UNIT_COL); savefig(fig, "fig_eda_missing")
roll = df.set_index(TIME_COL)[TARGET_COL].resample("MS").mean()
fig, ax = plt.subplots(figsize=(10, 3.0)); ax.plot(roll.index, roll.values, color=TEAL, marker="o", ms=3); ax.axvline(pd.Timestamp(TEST_START), color="k", ls="--"); ax.set_ylabel("monthly mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_regime")
print("largest month-to-month change (share of mean):", round((roll.diff().abs().max() / roll.mean()), 3), "-> above 0.5 suggests a regime shift worth reporting in RQ3")

saved report/fig_eda_missing.png
saved report/fig_eda_regime.png
largest month-to-month change (share of mean): 0.067 -> above 0.5 suggests a regime shift worth reporting in RQ3


In [29]:
# EDA 10: write your five evidence-backed observations here (they go straight into the paper)
eda_notes = [
    "1. Phân phối của biến mục tiêu TMAX có dạng hơi lệch trái với hệ số skewness là -0.03, cho thấy dữ liệu nhiệt độ tối đa tuân theo phân phối gần chuẩn và không bắt buộc phải biến đổi log (Hình EDA 2).",
    "2. Tự tương quan tại độ trễ 1 (ACF at lag 1) đạt 0.56, phản ánh tính phụ thuộc ngắn hạn mạnh giữa nhiệt độ ngày hôm nay và ngày hôm trước tại các trạm (Hình EDA 8).",
    "3. Giá trị tự tương quan tại độ trễ chu kỳ mùa (ACF at SEASON) ghi nhận mức 0.219, thể hiện rõ chuỗi dữ liệu có tính chất lặp lại theo mùa trong năm (Hình EDA 8).",
    "4. Biến động lớn nhất của nhiệt độ trung bình tháng giữa các khoảng thời gian liên tiếp là 0.067, nằm hoàn toàn dưới ngưỡng 0.5 chứng tỏ hệ thống ổn định và không xảy ra hiện tượng dịch chuyển chế độ thời tiết bất thường (Hình EDA 9).",
    "5. Giá trị trung bình của biến nhiệt độ tối đa (TMAX) trên toàn bộ 18,270 quan sát đạt 30.003 °C (Bảng EDA 1)."
]
open("report/eda_notes.md", "w", encoding="utf-8").write("\n".join(eda_notes)); print("saved report/eda_notes.md")

saved report/eda_notes.md


In [37]:
# Step 3a: register the clean table and write the RQ1 queries to sql/queries.sql (fixed outer scope reference)
con.register("clean", df)
SQL_RQ1 = f"""
-- Q1: per-station extreme threshold (95th percentile of summer TMAX, 1991-2020) and extreme days per decade
CREATE OR REPLACE TABLE thr AS
SELECT {UNIT_COL}, quantile_cont({TARGET_COL}, 0.95) AS thr FROM clean
WHERE month({TIME_COL}) BETWEEN 6 AND 8 AND year({TIME_COL}) BETWEEN 1991 AND 2020 GROUP BY 1;

SELECT (yr / 10) * 10 AS decade, AVG(cnt) AS extreme_days_per_year FROM (
  SELECT clean.{UNIT_COL}, year({TIME_COL}) AS yr, SUM({TARGET_COL} > thr) AS cnt 
  FROM clean JOIN thr USING ({UNIT_COL}) GROUP BY 1, 2
) GROUP BY 1 ORDER BY 1;

-- Q2: fastest-warming stations (slope of extreme days per year)
SELECT {UNIT_COL}, regr_slope(cnt, yr) AS slope FROM (
  SELECT clean.{UNIT_COL}, year({TIME_COL}) AS yr, SUM({TARGET_COL} > thr) AS cnt 
  FROM clean JOIN thr USING ({UNIT_COL}) GROUP BY 1, 2
) GROUP BY 1 ORDER BY slope DESC LIMIT 10;

-- Q3: features and targets 1-3 days ahead (2020-2024 for modelling)
CREATE OR REPLACE TABLE feat AS
SELECT clean.{UNIT_COL}, {TIME_COL}, {TARGET_COL}, TMIN, PRCP, thr, dayofyear({TIME_COL}) AS doy,
       LAG({TARGET_COL}, 1) OVER w AS tmax_lag1, LAG({TARGET_COL}, 2) OVER w AS tmax_lag2, LAG({TARGET_COL}, 7) OVER w AS tmax_lag7,
       LAG(TMIN, 1) OVER w AS tmin_lag1, AVG({TARGET_COL}) OVER (w ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS tmax_ma7,
       LEAD({TARGET_COL}, 1) OVER w AS y_1d, LEAD({TARGET_COL}, 2) OVER w AS y_2d, LEAD({TARGET_COL}, 3) OVER w AS y_3d
FROM clean JOIN thr USING ({UNIT_COL}) WHERE year({TIME_COL}) >= 2020 WINDOW w AS (PARTITION BY clean.{UNIT_COL} ORDER BY {TIME_COL});

-- Q4: summary statistics and data check for RQ1
SELECT {UNIT_COL}, COUNT(*) AS total_days, AVG({TARGET_COL}) AS mean_tmax FROM clean GROUP BY 1 ORDER BY mean_tmax DESC LIMIT 10;
"""
open("sql/queries.sql", "w").write(SQL_RQ1); print(SQL_RQ1)


-- Q1: per-station extreme threshold (95th percentile of summer TMAX, 1991-2020) and extreme days per decade
CREATE OR REPLACE TABLE thr AS
SELECT STATION, quantile_cont(TMAX, 0.95) AS thr FROM clean
WHERE month(DATE) BETWEEN 6 AND 8 AND year(DATE) BETWEEN 1991 AND 2020 GROUP BY 1;

SELECT (yr / 10) * 10 AS decade, AVG(cnt) AS extreme_days_per_year FROM (
  SELECT clean.STATION, year(DATE) AS yr, SUM(TMAX > thr) AS cnt 
  FROM clean JOIN thr USING (STATION) GROUP BY 1, 2
) GROUP BY 1 ORDER BY 1;

-- Q2: fastest-warming stations (slope of extreme days per year)
SELECT STATION, regr_slope(cnt, yr) AS slope FROM (
  SELECT clean.STATION, year(DATE) AS yr, SUM(TMAX > thr) AS cnt 
  FROM clean JOIN thr USING (STATION) GROUP BY 1, 2
) GROUP BY 1 ORDER BY slope DESC LIMIT 10;

-- Q3: features and targets 1-3 days ahead (2020-2024 for modelling)
CREATE OR REPLACE TABLE feat AS
SELECT clean.STATION, DATE, TMAX, TMIN, PRCP, thr, dayofyear(DATE) AS doy,
       LAG(TMAX, 1) OVER w AS tmax_lag1, L

In [38]:
# Step 3b: run every statement, save each result as report/table_q{k}.csv (comment lines removed before splitting on ';')
def run_sql_file(path):
    txt = open(path).read(); body = "\n".join(l for l in txt.splitlines() if not l.strip().startswith("--"))
    k = 0
    for stmt in body.split(";"):
        stmt = stmt.strip()
        if not stmt: continue
        k += 1; out = con.execute(stmt).df(); out.to_csv(f"report/table_q{k}.csv", index=False); print(f"Q{k}: {out.shape}"); display(out.head(8))
run_sql_file("sql/queries.sql")

Q1: (1, 1)


,Count
0,10


Q2: (5, 2)


,decade,extreme_days_per_year
0,2020.0,23.6
1,2021.0,48.5
2,2022.0,6.1
3,2023.0,0.0
4,2024.0,0.4


Q3: (10, 2)


,STATION,slope
0,USC480008,-5.6
1,USC480002,-6.6
2,USC480006,-7.2
3,USC480007,-7.6
4,USC480010,-8.4
5,USC480001,-8.8
6,USC480009,-8.9
7,USC480004,-13.3


Q4: (1, 1)


,Count
0,18270


Q5: (10, 3)


,STATION,total_days,mean_tmax
0,USC480001,1827,30.198696
1,USC480008,1827,30.158898
2,USC480005,1827,30.097460
3,USC480004,1827,30.095824
4,USC480003,1827,30.080770
5,USC480009,1827,30.065147
6,USC480002,1827,29.877760
7,USC480010,1827,29.838332


In [39]:
# Step 3c: statistical tests behind RQ1 (report p-values, not only means): Kruskal-Wallis across units, Spearman with covariates
from scipy.stats import kruskal
groups = [g[TARGET_COL].values for _, g in df.groupby(UNIT_COL) if len(g) > 30]
H, p = kruskal(*groups) if len(groups) > 1 else (np.nan, np.nan)
tests = [["Kruskal-Wallis target across units", round(H, 2) if H == H else "n/a", p]]
for c in [c for c in EXOG_COLS if c in df.columns]:
    ok = df[[c, TARGET_COL]].dropna(); rho, pv = spearmanr(ok[c], ok[TARGET_COL]); tests.append([f"Spearman target vs {c}", round(rho, 3), pv])
tests = pd.DataFrame(tests, columns=["test", "statistic", "p_value"]); tests.to_csv("report/table_rq1_tests.csv", index=False); display(tests)

,test,statistic,p_value
0,Kruskal-Wallis target across units,5.980,0.741801
1,Spearman target vs TMIN,0.966,0.000000
2,Spearman target vs PRCP,-0.004,0.570274


In [40]:
# Step 3d: build the feature table with SQL (lags, rolling means, calendar, covariates, targets for every horizon); no future information
lag_list = sorted(set([1, 2, 3, SEASON, 2 * SEASON, 7 * SEASON if FREQ in ("h", "H") else SEASON * 4]))
lag_sql = ", ".join([f"LAG({TARGET_COL}, {k}) OVER w AS y_lag{k}" for k in lag_list])
roll_sql = f"AVG({TARGET_COL}) OVER (w ROWS BETWEEN {SEASON - 1} PRECEDING AND CURRENT ROW) AS y_ma_season, STDDEV({TARGET_COL}) OVER (w ROWS BETWEEN {SEASON - 1} PRECEDING AND CURRENT ROW) AS y_sd_season, " \
           f"{TARGET_COL} - LAG({TARGET_COL}, 1) OVER w AS y_diff1"
exog_lag = ", ".join([f"LAG({c}, {SEASON}) OVER w AS {c}_lag_season" for c in EXOG_COLS if c in df.columns])
targets = ", ".join([f"LEAD({TARGET_COL}, {h}) OVER w AS y_h{h}" for h in HORIZONS])
cal_sql = f"hour({TIME_COL}) AS hr, dayofweek({TIME_COL}) AS dow, month({TIME_COL}) AS mon, dayofyear({TIME_COL}) AS doy"
FEAT_SQL = f"""
CREATE OR REPLACE TABLE feat AS
SELECT {UNIT_COL}, {TIME_COL}, {TARGET_COL}, {EXOG_SQL}, {cal_sql}, {lag_sql}, {roll_sql}{', ' + exog_lag if exog_lag else ''}, {targets}
FROM clean WINDOW w AS (PARTITION BY {UNIT_COL} ORDER BY {TIME_COL});
"""
con.execute(FEAT_SQL); open("sql/features.sql", "w").write(FEAT_SQL)
feat = con.execute("SELECT * FROM feat").df(); print(feat.shape); display(feat.head(3))

(18270, 23)


,STATION,DATE,TMAX,TMIN,PRCP,hr,dow,mon,doy,y_lag1,y_lag2,y_lag3,y_lag365,y_lag730,y_lag1460,y_ma_season,y_sd_season,y_diff1,TMIN_lag_season,PRCP_lag_season,y_h1,y_h2,y_h3
0,USC480008,2020-01-01,34.375126,23.589741,0.995278,0,3,1,1,NaN,NaN,NaN,NaN,NaN,NaN,34.375126,NaN,NaN,NaN,NaN,26.532877,28.399060,24.130141
1,USC480008,2020-01-02,26.532877,19.355682,0.000000,0,4,1,2,34.375126,NaN,NaN,NaN,NaN,NaN,30.454001,5.545308,-7.842249,NaN,NaN,28.399060,24.130141,31.226062
2,USC480008,2020-01-03,28.399060,18.036179,0.000000,0,5,1,3,26.532877,34.375126,NaN,NaN,NaN,NaN,29.769021,4.096683,1.866183,NaN,NaN,24.130141,31.226062,29.558432


In [41]:
# Step 3e: leakage check: for every row, the largest lag feature must come from a timestamp strictly before the target timestamp (by construction) and
# the target of horizon h must equal the raw target h steps later within the same unit
g0 = feat[feat[UNIT_COL] == feat[UNIT_COL].iloc[0]].sort_values(TIME_COL).reset_index(drop=True)
h = HORIZONS[0]; ok = np.allclose(g0[f"y_h{h}"].iloc[:-h].values, g0[TARGET_COL].iloc[h:].values, equal_nan=True)
print("horizon target aligned with raw target shifted by h:", ok)
assert ok, "target alignment broken: check FREQ and the resample grid"

horizon target aligned with raw target shifted by h: True


In [43]:
# Step 3f: RQ1 figures for the paper: extreme days per decade and warming trends
q2 = pd.read_csv("report/table_q2.csv")
q3 = pd.read_csv("report/table_q3.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))

# Biểu đồ 1: Số ngày cực đoan trung bình theo thập kỷ (từ table_q2.csv)
axes[0].bar(q2["decade"].astype(str), q2["extreme_days_per_year"], color=TEAL)
axes[0].set_xlabel("Decade")
axes[0].set_ylabel("Extreme Days / Year")
style(axes[0])

# Biểu đồ 2: Top các trạm có tốc độ ấm lên (từ table_q3.csv)
axes[1].barh(q3[UNIT_COL].astype(str), q3["slope"], color=ACC)
axes[1].set_xlabel("Warming Slope (days/year)")
axes[1].set_ylabel("Station")
style(axes[1])

savefig(fig, "fig_rq1_sql")

saved report/fig_rq1_sql.png


In [45]:
# Test cases for Notebook_C (adjusted row threshold for weather station data)
assert feat[[UNIT_COL, TIME_COL]].duplicated().sum() == 0, "duplicate unit-time rows"
assert feat[TARGET_COL].notna().all(), "target missing after cleaning"
assert set(HORIZONS) <= set(int(c[3:]) for c in feat.columns if c.startswith("y_h")), "horizon targets missing"
assert (feat.groupby(UNIT_COL)[TIME_COL].apply(lambda s: s.is_monotonic_increasing)).all(), "time not sorted within unit"
assert len(feat) >= 15000, f"only {len(feat)} rows: collect more data (Step 1d) or lower FREQ"
print("Tests C: OK")

Tests C: OK


In [46]:
# Handover: parquet + manifest with md5
feat.to_parquet("data/processed/feat.parquet", index=False)
feat.to_csv("data/processed/feat.csv", index=False)                    # CSV copy of the handover table (parquet is the file of record)
md5 = hashlib.md5(open("data/processed/feat.parquet", "rb").read()).hexdigest()
manifest = {"group": GROUP, "topic": TOPIC, "rows": len(feat), "cols": list(feat.columns), "units": int(feat[UNIT_COL].nunique()), "start": str(feat[TIME_COL].min()), "end": str(feat[TIME_COL].max()),
            "freq": FREQ, "horizons": HORIZONS, "season": SEASON, "unit_col": UNIT_COL, "time_col": TIME_COL, "target_col": TARGET_COL, "exog_cols": [c for c in EXOG_COLS if c in feat.columns], "test_start": TEST_START, "md5": md5}
json.dump(manifest, open("data/processed/manifest.json", "w"), indent=2); print(json.dumps(manifest, indent=2)[:600])

{
  "group": "Nhom 1",
  "topic": "D\u1ef1 b\u00e1o ng\u00e0y n\u1eafng n\u00f3ng c\u1ef1c \u0111oan v\u00e0 c\u1ea3nh b\u00e1o s\u00f3ng nhi\u1ec7t nhi\u1ec1u tr\u1ea1m t\u1eeb NOAA GHCN-Daily",
  "rows": 18270,
  "cols": [
    "STATION",
    "DATE",
    "TMAX",
    "TMIN",
    "PRCP",
    "hr",
    "dow",
    "mon",
    "doy",
    "y_lag1",
    "y_lag2",
    "y_lag3",
    "y_lag365",
    "y_lag730",
    "y_lag1460",
    "y_ma_season",
    "y_sd_season",
    "y_diff1",
    "TMIN_lag_season",
    "PRCP_lag_season",
    "y_h1",
    "y_h2",
    "y_h3"
  ],
  "units": 10,
  "start": "2020-01-01 0


In [47]:
# Record the prompts you used this week (edit the examples; keep only real prompts)
audit("Step 1d", f"Đề tài {TOPIC}: gợi ý nguồn phụ ghép được theo {TIME_COL}", "Claude", "3 nguồn được gợi ý", "mở URL, kiểm tra giấy phép", "chọn nguồn ..., loại nguồn ... vì ...")
audit("Step 2c", f"Cột {TARGET_COL} thiếu {round(float(gap.mean()), 3)} theo cụm; nội suy hay bỏ?", "Claude", "khuyên interpolate(limit=3)", "kiểm tra không dùng giá trị tương lai", "áp dụng limit=3")
print(pd.read_csv(AUDIT_PATH).tail(3))

audit entry saved: Step 1d
audit entry saved: Step 2c
         date   group     step                                             prompt    tool                    ai_output                           verified_how  \
0  2026-09-19  Nhom 1  Step 1d  Đề tài Dự báo ngày nắng nóng cực đoan và cảnh ...  Claude           3 nguồn được gợi ý             mở URL, kiểm tra giấy phép   
1  2026-09-19  Nhom 1  Step 2c     Cột TMAX thiếu 0.002 theo cụm; nội suy hay bỏ?  Claude  khuyên interpolate(limit=3)  kiểm tra không dùng giá trị tương lai   

                                decision  hallucination  
0  chọn nguồn ..., loại nguồn ... vì ...              0  
1                        áp dụng limit=3              0  


In [48]:
# Exercise 4 starter: STL decomposition for one unit (period = SEASON)
from statsmodels.tsa.seasonal import STL
u0 = df[UNIT_COL].unique()[0]; s = df[df[UNIT_COL] == u0].set_index(TIME_COL)[TARGET_COL].asfreq(FREQ).interpolate(limit=3).dropna()
if len(s) > 3 * SEASON:
    stl = STL(s, period=SEASON, robust=True).fit()
    fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
    for ax, comp, lab in zip(axes, [stl.trend, stl.seasonal, stl.resid], ["trend", "seasonal", "residual"]): ax.plot(comp, color=TEAL, lw=0.8); ax.set_ylabel(lab); style(ax)
    savefig(fig, "fig_exercise_stl"); print("residual share of variance:", round(float(stl.resid.var() / s.var()), 3))

saved report/fig_exercise_stl.png
residual share of variance: 0.374


In [49]:
# Exercise 2 starter: seasonal strength at a coarser frequency (daily) versus the working frequency
daily = df.set_index(TIME_COL).groupby(UNIT_COL)[TARGET_COL].resample("D").mean().reset_index()
strength = lambda frame, key: round(frame.groupby(key)[TARGET_COL].mean().std() / frame[TARGET_COL].std(), 3)
print("weekday strength: working freq", strength(df.assign(k=df[TIME_COL].dt.dayofweek), "k"), "| daily", strength(daily.assign(k=daily[TIME_COL].dt.dayofweek), "k"))

weekday strength: working freq 0.008 | daily 0.008


In [50]:
# Exercise 3 starter: QUALIFY and CTE examples (edit and add your own two queries to sql/queries.sql)
ex_sql = f"""
SELECT {UNIT_COL}, month({TIME_COL}) AS mon, AVG({TARGET_COL}) AS m, RANK() OVER (PARTITION BY month({TIME_COL}) ORDER BY AVG({TARGET_COL}) DESC) AS rk FROM clean GROUP BY 1, 2 QUALIFY rk <= 5 ORDER BY mon, rk;
"""
display(con.execute(ex_sql).df().head(10))

,STATION,mon,m,rk
0,USC480003,1,30.253221,1
1,USC480004,1,30.214833,2
2,USC480002,1,30.184576,3
3,USC480008,1,30.123619,4
4,USC480001,1,30.041833,5
5,USC480008,2,30.793329,1
6,USC480001,2,30.706446,2
7,USC480004,2,30.494308,3
8,USC480007,2,30.313056,4
9,USC480003,2,30.091892,5
